
&Auml;
Eine `View`-Objekt beobachtet das `Game`-Objekt.



Die Methode`observe(fun)` registriert die Funktion `fun` unter ihrem Namen als Callback.
Sie nimmt das Schlüssel-Wert Paar `fun.__name__`, `fun` in den Dict `callbacks` auf.
Die Methode `notify(event, **kwargs)` ruft alle registrierten Callbacks `fun` mit den
Argumenten `fun(event, data)` auf.





Ein **Observable**, das in irgendeiner Form dargestellt werden möchte, kann z.B.
nach relevanten Zustandsänderungen jeweils `notify('push_stone', old=(1, 1), new=(2, 3))`
aufrufen. Ein (beoachtendes) Objekt, welches  verantwortlich ist für die graphische Darstellung des Zustands des Observables, kann dann 
ein Callback registrieren, welches aufgrund der Informationen `push_stone` und den Keyword Argumenten 
`old=(1, 1)`, `new=(2, 3)` die Darstellung updaten kann.
Oft hat auch das beoachtende Objekt Zugriff auf das Observable. Dann genügt u.U. schon die
Informationen `event` um Darstellung updaten zu können, jedoch kann zusätzliche Information die Aufgabe erleichtern.



Der Spieler kann den Stein mit einer Funktion `push_stone((dx, dy))` verschieben,
von der aktuellen Positon `(x, y)` nach `(x+dx, y+dy)`.
Rufen wir dann ein registriertes Callback `f`d er View-Komponente mit den Argumenten
`f('push_stone', (x, y), (x+dx, y+dy))` auf,
kann diese den Spielzustand updaten indem sie einfach das Feld `(x, y)` leert und den Stein neu
auf dem Feld `(x+dx, y+dy)` des Gitters zeichnet.

In [ ]:
class Game:
    def __init__(self):
        self.callbacks = []
        self.grid_size = 10
        self.games_played = 0
        self.stones = set()

    def observe(self, fun):
        if fun.__name__ not in self.callbacks:
            self.callbacks[f.__name__] = fun

    def unobserve(self, fun):
        if fun.__name__ in self.callbacks:
            self.callbacks.pop(fun.__name__) 

    def notify(self, event, **kwargs):
        for f in self.callbacks.values():
            f(event, **kwargs)

    def new_game(self):
        self.games_played += 1
        self.stones.clear()
        self.notify('newgame', games_played=self.games_played)

    def is_inside(self, pos):
        return (0 <= pos[0] < self.grid_size
                and 0 <= pos[1] < self.grid_size)

    def place_stone(self, pos):
        if not self.is_inside(pos) or pos in self.stones:
            self.notify('error',
                        msg=f'playing at Positon {pos} is illegal',
                        )
            return

        self.stones.add(pos)
        self.notify('place_stone', pos=pos)

    def move_stone(self, old, new):
        if new in self.stones or old not in self.stones:
            self.notify('error',
                        msg=f'cannot move from {old} to {new}!',
                        )
            return

        self.stones.remove(old)
        self.stones.add(new)
        self.notify('move_stone', old=old, new=new)

    def push_stone(self, pos, dpos):
         if new in self.stones or old not in self.stones:
            self.notify('error',
                        msg=f'cannot move from {old} to {new}!',
                        )
            return
        
        dx, dy = dpos
        

    def __repr__(self):
        return f'Games played: {self.games_played}, grid size: {self.grid_size},\n'\
               f'stones: {self.stones}'

In [ ]:
import canvas_helpers as H
from ipycanvas import Canvas
from ipywidgets import Output
from IPython.display import display


err_out = Output(layout={'border': '1px solid black'})


class CV:
    def __init__(self, game, boardspec=(20, 20, 20, 20, 8, 8)):
        self.game = game
        self.boardspec = boardspec

        self.canvas = Canvas(width=200, height=200, layout={'border': '1px solid black'})
        self.canvas.on_mouse_down(self.on_mouse_down)
        self.canvas.on_mouse_up(self.on_mouse_up)

        self.out = Output(layout={'border': '1px solid black'})
        H.draw_grid(self.canvas, boardspec, line_width=2, color='blue')

        self.click_pos = None

    @err_out.capture(clear_output=True)
    def on_mouse_down(self, x, y):
        self.click_pos = H.xy2cr(x, y, self.boardspec)

    @err_out.capture(clear_output=True)
    def on_mouse_up(self, x, y):
        release_pos = H.xy2cr(x, y, self.boardspec)
        if self.click_pos != release_pos:
            self.game.move_stone(self.click_pos, release_pos)
        else:
            self.game.place_stone(release_pos)
        self.click_pos = None

    def _ipython_display_(self):
        display(self.canvas, self.out, err_out)

In [ ]:
import canvas_helpers as H
from ipycanvas import Canvas
from ipywidgets import Output
from IPython.display import display


err_out = Output(layout={'border': '1px solid black'})


class CV1:
    def __init__(self, game, boardspec=(20, 20, 20, 20, 8, 8)):
        self.game = game
        self.boardspec = boardspec

        self.canvas = Canvas(width=20, height=20, layout={'border': '1px solid black'})
        self.canvas.on_key_down(self.on_key_down)
       

        self.out = Output(layout={'border': '1px solid black'})
       

       

    @err_out.capture(clear_output=True)
    def on_key_down(self, key, *flags):
        key_direction = {'ArrowUp': (0, -1),
                         'ArrowDown': (0, 1),
                         'ArrowRight': (1, 0),
                         'ArrowLeft': (-1, 0),
                         }
        if key in key_direction:
            dpos = key_direction[key]
            game.move_stone(dpos)

    def _ipython_display_(self):
        display(self.canvas, self.out, err_out)